# VQE Grid Search

## Setup

In [ ]:
import pandas as pd
from qiskit.primitives import StatevectorEstimator

from src.data import summarize_data_dir
from src.vqe.molecular_system import statevector_grid_systems
from src.vqe.grid_search import run_vqe_grid_search

pd.set_option("display.max_columns", None)
seed = 137

## Sistemas e Grade

Este notebook usa um perfil maior para comparar VQE com FCI em mais distancias, bases, ansatz e otimizadores. Para testar rapidamente uma mudanca de codigo, troque `grid_profile` para `"pilot"`.

In [ ]:
grid_profile = "full"  # use "pilot" for a quick smoke test
systems = statevector_grid_systems(profile=grid_profile)

parameter_grid = {
    "mapper": ["jw"],
    "ansatz": ["real_amplitudes", "efficient_su2"],
    "reps": [1, 2],
    "optimizer": ["cobyla", "slsqp", "spsa"],
    "max_iter": [100],
    "seed": [seed],
}

total_jobs = sum(len(system.distances) for system in systems)
total_jobs *= len(parameter_grid["mapper"])
total_jobs *= len(parameter_grid["ansatz"])
total_jobs *= len(parameter_grid["reps"])
total_jobs *= len(parameter_grid["optimizer"])
total_jobs *= len(parameter_grid["max_iter"])
total_jobs *= len(parameter_grid["seed"])

print(f"Systems: {len(systems)}")
print(f"VQE jobs: {total_jobs}")
[(system.name, system.basis, len(system.distances), system.active_space) for system in systems]

## Execucao Sem Ruido

In [ ]:
estimator = StatevectorEstimator()

results_df = run_vqe_grid_search(
    systems=systems,
    parameter_grid=parameter_grid,
    estimator=estimator,
    cache=True,
    overwrite=False,
    include_fci_reference=True,
)

results_df

### Resumo

In [ ]:
summary_cols = [
    "run_label",
    "backend_name",
    "molecule",
    "basis",
    "distance",
    "mapper",
    "ansatz",
    "reps",
    "optimizer",
    "energy",
    "reference_method",
    "reference_energy",
    "fci_energy",
    "abs_error_kcal_mol",
    "within_chemical_accuracy",
    "eval_count",
    "num_qubits",
    "num_terms",
    "success",
]

results_df[summary_cols].sort_values(["molecule", "distance", "reps"])

### Melhores Configuracoes

In [ ]:
best_cols = [
    "molecule",
    "basis",
    "distance",
    "ansatz",
    "reps",
    "optimizer",
    "energy",
    "reference_method",
    "reference_energy",
    "fci_energy",
    "abs_error_kcal_mol",
    "within_chemical_accuracy",
    "eval_count",
]

best_by_point = (
    results_df[results_df["success"]]
    .sort_values("abs_error_kcal_mol")
    .groupby(["molecule", "basis", "distance"], as_index=False)
    .head(1)
    .sort_values(["molecule", "basis", "distance"])
)

best_by_point[best_cols]

### Precisao Quimica

In [ ]:
accuracy_summary = (
    best_by_point
    .groupby(["molecule", "basis"], as_index=False)
    .agg(
        points=("distance", "count"),
        points_within_chemical_accuracy=("within_chemical_accuracy", "sum"),
        mean_abs_error_kcal_mol=("abs_error_kcal_mol", "mean"),
        max_abs_error_kcal_mol=("abs_error_kcal_mol", "max"),
    )
)

accuracy_summary["chemical_accuracy_rate"] = (
    accuracy_summary["points_within_chemical_accuracy"] / accuracy_summary["points"]
)

accuracy_summary.sort_values(["molecule", "basis"])

### Inventario Do Cache

In [ ]:
pd.DataFrame(summarize_data_dir()).sort_values(["molecule", "basis", "kind", "extension"])